In [0]:
storage_path = "abfss://raw@deltravels.dfs.core.windows.net/bus/"
display(dbutils.fs.ls(storage_path))

path,name,size,modificationTime
abfss://raw@deltravels.dfs.core.windows.net/bus/agency.txt,agency.txt,260,1792651791000
abfss://raw@deltravels.dfs.core.windows.net/bus/calendar.txt,calendar.txt,122,1792651791000
abfss://raw@deltravels.dfs.core.windows.net/bus/routes.txt,routes.txt,52002,1792651873000
abfss://raw@deltravels.dfs.core.windows.net/bus/stop_times.txt,stop_times.txt,142098714,1792652884000
abfss://raw@deltravels.dfs.core.windows.net/bus/stops.txt,stops.txt,612802,1792651872000
abfss://raw@deltravels.dfs.core.windows.net/bus/trips.txt,trips.txt,1788264,1792651928000


In [0]:

routes = spark.read.csv("abfss://raw@deltravels.dfs.core.windows.net/bus/routes.txt", header=True, inferSchema=True)
stops = spark.read.csv("abfss://raw@deltravels.dfs.core.windows.net/bus/stops.txt", header=True, inferSchema=True)
trips = spark.read.csv("abfss://raw@deltravels.dfs.core.windows.net/bus/trips.txt", header=True, inferSchema=True)

print(routes.count())
print(stops.count())
print(trips.count())

display(routes.limit(5))

2403
10559
89393


agency_id,route_id,route_long_name,route_short_name,route_type
DIMTS,142,828AUP,null,3
DIMTS,10001,971DOWN,null,3
DIMTS,362,961DOWN,null,3
DIMTS,274,819DOWN,null,3
DIMTS,10386,824STLDOWN2,null,3


Data ingestion (Bronze)

In [0]:
spark.sql("USE CATALOG deltraveldata")
spark.sql("USE SCHEMA default")

DataFrame[]

In [0]:

routes.write.format("delta").mode("overwrite").saveAsTable("bronze_routes")
stops.write.format("delta").mode("overwrite").saveAsTable("bronze_stops")
trips.write.format("delta").mode("overwrite").saveAsTable("bronze_trips")

In [0]:
routes = spark.table("bronze_routes")
routes.printSchema()
display(routes.limit(10))

root
 |-- agency_id: string (nullable = true)
 |-- route_id: integer (nullable = true)
 |-- route_long_name: string (nullable = true)
 |-- route_short_name: string (nullable = true)
 |-- route_type: integer (nullable = true)



agency_id,route_id,route_long_name,route_short_name,route_type
DIMTS,142,828AUP,null,3
DIMTS,10001,971DOWN,null,3
DIMTS,362,961DOWN,null,3
DIMTS,274,819DOWN,null,3
DIMTS,10386,824STLDOWN2,null,3
DIMTS,233,919UP,null,3
DIMTS,10387,770ASTLDOWN2,null,3
DIMTS,118,817NSTLDOWN,null,3
DIMTS,10388,73ADOWN,null,3
DIMTS,67,112STLUP,null,3


In [0]:
print("Total routes:", routes.count())
print("Unique routes:", routes.select("route_id").distinct().count())



Total routes: 2403
Unique routes: 2403


In [0]:
print("Missing route IDs:", routes.filter("route_id IS NULL").count())
print("Missing agency IDs:", routes.filter("agency_id IS NULL").count())

Missing route IDs: 0
Missing agency IDs: 0


SILVER


Routes


In [0]:
silver_routes = routes.dropDuplicates(["route_id"])
silver_routes.write.format("delta").mode("overwrite").saveAsTable("silver_routes")
print("Silver routes:", silver_routes.count())


Silver routes: 2403


In [0]:
##since route_short_name is all null will drop it
silver_routes = routes.drop("route_short_name")
silver_routes.write.format("delta").mode("overwrite").saveAsTable("silver_routes")

In [0]:
print(silver_routes.columns)
print("Total rows:", silver_routes.count())


['agency_id', 'route_id', 'route_long_name', 'route_type']
Total rows: 2403


STOPS

In [0]:
##To check the count if total stops,unique stop ids and missing stop ids
stops = spark.table("bronze_stops")

print("Total stops:", stops.count())
print("Unique stop IDs:", stops.select("stop_id").distinct().count())
print("Missing stop IDs:", stops.filter("stop_id IS NULL").count())

Total stops: 10559
Unique stop IDs: 10559
Missing stop IDs: 0


In [0]:
## To Check whether stop names and location coordinates are missing from the dataset.

print("Missing stop names:", stops.filter("stop_name IS NULL").count())
print("Missing latitude:", stops.filter("stop_lat IS NULL").count())
print("Missing longitude:", stops.filter("stop_lon IS NULL").count())

Missing stop names: 0
Missing latitude: 0
Missing longitude: 0


In [0]:
silver_stops = stops.dropDuplicates(["stop_id"])
silver_stops.write.format("delta").mode("overwrite").saveAsTable("silver_stops")

TRIPS

In [0]:
## check how many scheduled trips we have, whether trip IDs are unique, and whether any trip IDs are missing

trips = spark.table("bronze_trips")

print("Total trips:", trips.count())
print("Unique trip IDs:", trips.select("trip_id").distinct().count())
print("Missing trip IDs:", trips.filter("trip_id IS NULL").count())



Total trips: 89393
Unique trip IDs: 89393
Missing trip IDs: 0


In [0]:
print("Missing route IDs:", trips.filter("route_id IS NULL").count())
print("Missing service IDs:", trips.filter("service_id IS NULL").count())

Missing route IDs: 0
Missing service IDs: 0


In [0]:
silver_trips = trips.dropDuplicates(["trip_id"])
silver_trips.write.format("delta").mode("overwrite").saveAsTable("silver_trips")


STOP TIMES

In [0]:
stop_times = spark.read.csv("abfss://raw@deltravels.dfs.core.windows.net/bus/stop_times.txt",
                            header=True,
                            inferSchema=True
)
print("Total stop times:", stop_times.count())
print("Missing trip IDs:", stop_times.filter("trip_id IS NULL").count())

Total stop times: 3724320
Missing trip IDs: 0


In [0]:
silver_stop_times = stop_times.dropDuplicates()
silver_stop_times.write.format("delta").mode("overwrite").saveAsTable("silver_stop_times")

In [0]:
print("Missing trip IDs:", stop_times.filter("trip_id IS NULL").count())
print("Missing arrival times:", stop_times.filter("arrival_time IS NULL").count())
print("Missing departure times:", stop_times.filter("departure_time IS NULL").count())
print("Missing stop IDs:", stop_times.filter("stop_id IS NULL").count())
print("Missing stop sequence:", stop_times.filter("stop_sequence IS NULL").count())

Missing trip IDs: 0
Missing arrival times: 0
Missing departure times: 0
Missing stop IDs: 0
Missing stop sequence: 0


In [0]:
stop_times.write.format("delta").mode("overwrite").saveAsTable("bronze_stop_times")

In [0]:
print("Routes:", silver_routes.columns)
print("Stops:", silver_stops.columns)
print("Trips:", silver_trips.columns)
print("Stop times:", silver_stop_times.columns)

Routes: ['agency_id', 'route_id', 'route_long_name', 'route_type']
Stops: ['stop_code', 'stop_id', 'stop_lat', 'stop_lon', 'stop_name', 'zone_id']
Trips: ['route_id', 'service_id', 'trip_id', 'shape_id']
Stop times: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence']


DATA MODELLING

dimension-route

In [0]:
dim_route = silver_routes.select(
    "route_id",
    "agency_id",
    "route_long_name",
    "route_type"
).dropDuplicates(["route_id"])

dim_route.write.format("delta").mode("overwrite").saveAsTable("dim_route")

In [0]:
print("Rows:", spark.table("dim_route").count())
print("Unique route IDs:", spark.table("dim_route").select("route_id").distinct().count())

Rows: 2403
Unique route IDs: 2403


dimension-stop

In [0]:
dim_stop = silver_stops.select(
    "stop_id",
    "stop_code",
    "stop_name",
    "stop_lat",
    "stop_lon",
    "zone_id"
).dropDuplicates(["stop_id"])

dim_stop.write.format("delta").mode("overwrite").saveAsTable("dim_stop")

In [0]:
print("Rows:", spark.table("dim_stop").count())
print("Unique stop IDs:", spark.table("dim_stop").select("stop_id").distinct().count())

Rows: 10559
Unique stop IDs: 10559


dimension trip

In [0]:
dim_trip = silver_trips.select(
    "trip_id",
    "route_id",
    "service_id",
    "shape_id"
).dropDuplicates(["trip_id"])

dim_trip.write.format("delta").mode("overwrite").saveAsTable("dim_trip")

In [0]:
print("Rows:", spark.table("dim_trip").count())
print("Unique trip IDs:", spark.table("dim_trip").select("trip_id").distinct().count())

Rows: 89393
Unique trip IDs: 89393


FACT TABLE


In [0]:
fact_trip_stop = silver_stop_times.select(
    "trip_id",
    "stop_id",
    "stop_sequence",
    "arrival_time",
    "departure_time"
)

fact_trip_stop.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_trip_stop")

print("Total rows:", spark.table("fact_trip_stop").count())

Total rows: 3724320


In [0]:
from pyspark.sql.functions import count

duplicate_keys = (
    spark.table("fact_trip_stop")
    .groupBy("trip_id", "stop_sequence")
    .count()
    .filter("count > 1")
)

print("Duplicate trip-stop keys:", duplicate_keys.count())

Duplicate trip-stop keys: 0


In [0]:
fact = spark.table("fact_trip_stop")

missing_trips = fact.join(
    spark.table("dim_trip").select("trip_id"),
    on="trip_id",
    how="left_anti"
).count()

missing_stops = fact.join(
    spark.table("dim_stop").select("stop_id"),
    on="stop_id",
    how="left_anti"
).count()

print("Fact rows with missing trips:", missing_trips)
print("Fact rows with missing stops:", missing_stops)

Fact rows with missing trips: 0
Fact rows with missing stops: 0


SQL ANALYTICS


Top 10 routes by scheduled trips

In [0]:
%sql
SELECT
    r.route_id,
    r.route_long_name,
    COUNT(t.trip_id) AS total_trips
FROM dim_route r
JOIN dim_trip t
    ON r.route_id = t.route_id
GROUP BY
    r.route_id,
    r.route_long_name
ORDER BY total_trips DESC
LIMIT 10;

route_id,route_long_name,total_trips
5721,623DOWN,318
5722,623ADOWN,318
674,628STLDOWN,313
593,628STLUP,311
5254,623UP,272
10533,182EUP,242
10088,GL-23DOWN,232
10067,GL-23UP,225
375,258UP,211
761,258DOWN,204


stops served by the most trips

In [0]:
%sql
SELECT
    s.stop_id,
    s.stop_name,
    COUNT(DISTINCT f.trip_id) AS total_trips
FROM dim_stop s
JOIN fact_trip_stop f
    ON s.stop_id = f.stop_id
GROUP BY
    s.stop_id,
    s.stop_name
ORDER BY total_trips DESC
LIMIT 10;

stop_id,stop_name,total_trips
149,Health Centre,3998
488,NajafGarh Delhi Gate,3996
233,Police Station NajafGarh,3787
10225,Jahangir Puri Metro Station,3653
10775,Super Bazar,3356
11475,Maharaj Pur Check Post,3301
10060,Anand Vihar ISBT Main Road,3277
10221,Adarsh Nagar / Bharola Village,3236
634,Uttam Nagar Terminal,3089
11044,Delhi Secretariat,3013


Average stops per trip

In [0]:
%sql
SELECT
    COUNT(*) AS total_stop_occurrences,
    COUNT(DISTINCT trip_id) AS total_trips,
    ROUND(
        COUNT(*) * 1.0 / COUNT(DISTINCT trip_id),
        2
    ) AS avg_stops_per_trip
FROM fact_trip_stop;

total_stop_occurrences,total_trips,avg_stops_per_trip
3724320,89393,41.66


In [0]:
%sql
WITH trip_stop_counts AS (
    SELECT
        trip_id,
        COUNT(*) AS stop_count
    FROM fact_trip_stop
    GROUP BY trip_id
)
SELECT
    COUNT(*) AS total_trips,
    MIN(stop_count) AS min_stops,
    ROUND(AVG(stop_count), 2) AS avg_stops,
    MAX(stop_count) AS max_stops,
    SUM(CASE WHEN stop_count < 5 THEN 1 ELSE 0 END) AS trips_under_5_stops,
    SUM(CASE WHEN stop_count > 100 THEN 1 ELSE 0 END) AS trips_over_100_stops
FROM trip_stop_counts;

total_trips,min_stops,avg_stops,max_stops,trips_under_5_stops,trips_over_100_stops
89393,2,41.66,197,841,276


In [0]:
%sql
WITH trip_stop_counts AS (
    SELECT
        trip_id,
        COUNT(*) AS stop_count
    FROM fact_trip_stop
    GROUP BY trip_id
)
SELECT
    t.route_id,
    r.route_long_name,
    COUNT(*) AS short_trip_count,
    MIN(c.stop_count) AS min_stops,
    MAX(c.stop_count) AS max_stops
FROM trip_stop_counts c
JOIN dim_trip t
    ON c.trip_id = t.trip_id
JOIN dim_route r
    ON t.route_id = r.route_id
WHERE c.stop_count < 5
GROUP BY
    t.route_id,
    r.route_long_name
ORDER BY short_trip_count DESC
LIMIT 10;

route_id,route_long_name,short_trip_count,min_stops,max_stops
10561,891LSTLDOWN,146,3,3
10014,S-1STLDOWN,104,4,4
259,215STLUP,81,3,3
1758,215STLDOWN,76,3,3
26,353STLDOWN,49,4,4
10450,972BSTLDOWN,40,3,3
10535,972BSTLUP,40,3,3
10398,136LNKSTLDWN2,39,4,4
437,784STLDOWN,25,3,3
1386,784STLUP,25,4,4


In [0]:
%sql
SELECT
    t.route_id,
    r.route_long_name,
    COUNT(DISTINCT t.trip_id) AS total_trips,
    COUNT(DISTINCT f.stop_id) AS distinct_stops_served
FROM dim_trip t
JOIN dim_route r
    ON t.route_id = r.route_id
JOIN fact_trip_stop f
    ON t.trip_id = f.trip_id
GROUP BY
    t.route_id,
    r.route_long_name
ORDER BY total_trips DESC
LIMIT 20;

route_id,route_long_name,total_trips,distinct_stops_served
5722,623ADOWN,318,61
5721,623DOWN,318,57
674,628STLDOWN,313,40
593,628STLUP,311,38
5254,623UP,272,58
10533,182EUP,242,20
10088,GL-23DOWN,232,25
10067,GL-23UP,225,24
375,258UP,211,25
761,258DOWN,204,27


In [0]:
%sql
SELECT
    r.route_id,
    r.route_long_name,
    COUNT(DISTINCT t.trip_id) AS total_trips,
    COUNT(DISTINCT f.stop_id) AS distinct_stops_served
FROM dim_route r
JOIN dim_trip t
    ON r.route_id = t.route_id
JOIN fact_trip_stop f
    ON t.trip_id = f.trip_id
GROUP BY
    r.route_id,
    r.route_long_name
ORDER BY distinct_stops_served DESC
LIMIT 10;

route_id,route_long_name,total_trips,distinct_stops_served
10553,(-)OUTERMUDRIKA,55,197
10040,OMS(-),12,196
10423,(+)OUTERMUDRIKA,35,176
10066,OMS(+),29,175
5428,OMS(+)UP,47,168
5895,OMS(-)DOWN,82,168
1646,701UP,8,102
962,701DOWN,8,102
5431,GRAMINMUDRIKA(+)UP,140,96
5898,GRAMINMUDRIKA(-)DOWN,110,96
